# Generating Synthetic Healthcare Data

This notebook shows how to **generate** synthetic healthcare data for research and development.

## Why Synthetic Data?

Real healthcare data is protected by HIPAA, GDPR, and other regulations. Synthetic data provides:

- **Privacy-safe** development and testing
- **No IRB approval** needed
- **Realistic patterns** for algorithm development
- **Unlimited scale** for benchmarking

## What We'll Cover

| Data Type | Method | Has Realistic Patterns? |
|-----------|--------|------------------------|
| **EHR** | Synthea | ✅ Yes (disease models, treatment pathways) |
| **Genomics** | HAPNEST (download) | ✅ Yes (linkage disequilibrium) |
| **Genomics** | HAPNESTRunner (generate) | ✅ Yes (linkage disequilibrium) |
| **Genomics** | Random genotypes | ❌ No (testing only!) |
| **Imaging** | Public datasets | ✅ Yes (real de-identified images) |

---

> **Already have data?** See [Using Multimodal Data](./Coherent_MultimodalDataset.ipynb) to work with the pre-generated Coherent Dataset.

In [19]:
%reload_ext autoreload
%autoreload 1 

import synthlab as sl 

---
## 1. Electronic Health Records (EHR)

### What's in an EHR?

An Electronic Health Record contains a patient's complete medical history:

| Component | Examples | Clinical Use |
|-----------|----------|-------------|
| **Demographics** | Age, gender, address | Population health, access to care |
| **Conditions** | Diabetes, hypertension | Diagnosis, disease management |
| **Medications** | Metformin, lisinopril | Treatment, drug interactions |
| **Labs/Vitals** | Blood glucose, blood pressure | Monitoring, diagnosis |
| **Procedures** | Surgery, imaging | Treatment history |
| **Encounters** | Office visits, ER visits | Care utilization |

### Synthea: Realistic Patient Simulation

[Synthea](https://github.com/synthetichealth/synthea) generates realistic patient histories using:
- **Disease progression models** based on medical literature
- **Treatment pathways** following clinical guidelines
- **Demographics** matching US Census data

In [14]:
# Option 1: Download pre-generated datasets
# Already converted to OMOP CDM (common data model used by OHDSI)

print("Pre-generated Synthea datasets on AWS:")
print("=" * 50)

datasets = sl.list_synthea_datasets()
for name in set(datasets):  # Remove duplicates
    print(f"  • {name}")

print()
print("Download with:")
print("  sl.download_dataset('synthea1k', output_dir='./data')")

Pre-generated Synthea datasets on AWS:
Listing available datasets in s3://synthea-omop/...
  • synthea100k
  • synthea1k
  • synthea23m
  • synthea23m
  • synthea100k
  • synthea1k

Download with:
  sl.download_dataset('synthea1k', output_dir='./data')


In [15]:
# Option 2: Generate fresh patients with Synthea
# Requires: Java 11+

print("Generate new synthetic patients:")
print("=" * 50)
print()
print("from synthlab import SyntheaRunner, SyntheaConfig")
print()
print("runner = SyntheaRunner()  # Downloads Synthea automatically")
print()
print("config = SyntheaConfig(")
print("    population_size=1000,    # Number of patients")
print("    state='Massachusetts',   # US state for demographics")
print("    seed=42,                 # Reproducibility")
print(")")
print()
print("result = runner.run(config)")
print()
print("# Convert to OMOP format")
print("sl.convert_synthea_to_omop(result.output_dir, 'omop_output')")

Generate new synthetic patients:

from synthlab import SyntheaRunner, SyntheaConfig

runner = SyntheaRunner()  # Downloads Synthea automatically

config = SyntheaConfig(
    population_size=1000,    # Number of patients
    state='Massachusetts',   # US state for demographics
    seed=42,                 # Reproducibility
)

result = runner.run(config)

# Convert to OMOP format
sl.convert_synthea_to_omop(result.output_dir, 'omop_output')


---
## 2. Genomics Data

### What's in Genomic Data?

Genomic data describes variations in DNA:

| Term | Meaning | Example |
|------|---------|--------|
| **SNP** | Single nucleotide polymorphism | rs1234567 |
| **Genotype** | Your two copies of a variant | AA, AG, or GG |
| **Allele** | One version of a variant | A or G |
| **MAF** | Minor allele frequency | 0.15 (15% of population) |

### Why Realistic Genetics Matter

Real genomes have **linkage disequilibrium (LD)** - nearby variants are correlated because they're inherited together on the same chromosome.

```
Chromosome segment:
                    LD block (inherited together)
                    ├──────────────────────────┤
Position:    100    200    300    400    500    600    700
                    SNP1   SNP2   SNP3          SNP4
                     ↑      ↑      ↑
                     └──────┼──────┘
                       Correlated!
```

**Random genotypes don't have LD**, which breaks methods like GWAS, PRS, and fine-mapping.

### Genomics Options in SynthLab

| Option | Has LD? | Use For |
|--------|---------|---------|
| **Download HAPNEST** | ✅ Yes | Production research (pre-generated, 1M+ individuals) |
| **HAPNESTRunner** | ✅ Yes | Generate custom datasets (requires Singularity/Docker) |
| **Random genotypes** | ❌ No | Quick pipeline testing only |

**References**:
- [HAPNEST Paper](https://academic.oup.com/bioinformatics/article/39/9/btad535/7255913)
- [HAPNEST Data (BioStudies)](https://www.ebi.ac.uk/biostudies/studies/S-BSST936)
- [HAPNEST Code (GitHub)](https://github.com/intervene-EU-H2020/synthetic_data)

In [ ]:
# Option 1: Download pre-generated HAPNEST data (REALISTIC LD)
# Best for most research use cases

from synthlab.hapnest import HAPNEST_PAPER_URL, HAPNEST_BIOSTUDIES_URL

print("HAPNEST: Realistic Synthetic Genotypes")
print("=" * 60)
print()
print("Pre-generated dataset with REALISTIC LD structure")
print("  • 1,008,000+ individuals")
print("  • 6.8 million variants")
print("  • 6 ancestry groups")
print()

info = sl.list_hapnest_files()
print("Ancestry groups:")
for code, name in info.get('ancestries', {}).items():
    print(f"  • {code}: {name}")

print()
print("Download with:")
print("  data_dir = sl.download_hapnest()")
print()
print(f"Paper: {HAPNEST_PAPER_URL}")
print(f"Data: {HAPNEST_BIOSTUDIES_URL}")

HAPNEST: Realistic Synthetic Genotypes

Pre-generated dataset with REALISTIC LD structure
  • 1,008,000+ individuals
  • 6.8 million variants
  • 6 ancestry groups

Ancestry groups:
  • AFR: African
  • AMR: Admixed American
  • CSA: Central/South Asian
  • EAS: East Asian
  • EUR: European
  • MID: Middle Eastern

Download with:
  data_dir = sl.download_hapnest()

Paper: https://academic.oup.com/bioinformatics/article/39/9/btad535/7255913
Data: https://www.ebi.ac.uk/biostudies/studies/S-BSST936


In [ ]:
# Option 2: Generate NEW realistic data with HAPNESTRunner
# Requires: Singularity or Docker container

from synthlab.hapnest import HAPNEST_GITHUB_URL

print("HAPNESTRunner: Generate Custom Realistic Genotypes")
print("=" * 60)
print()
print("Generate NEW synthetic genotypes with realistic LD structure.")
print("Requires Singularity or Docker (downloads ~10GB reference data).")
print()
print("from synthlab import HAPNESTRunner, HAPNESTConfig")
print()
print("# Initialize runner (downloads container + reference data)")
print("runner = HAPNESTRunner()")
print("runner.initialize()")
print()
print("# Configure generation")
print("config = HAPNESTConfig(")
print("    n_samples=1000,        # Number of individuals")
print("    chromosome=22,         # Single chromosome (1-22 or 'all')")
print("    superpopulation='EUR', # Ancestry: AFR, AMR, CSA, EAS, EUR, MID")
print("    seed=42,               # Reproducibility")
print(")")
print()
print("# Generate!")
print("result = runner.run(config)")
print(f"# Output in: {{result['output_dir']}}")
print()
print(f"Code: {HAPNEST_GITHUB_URL}")

HAPNESTRunner: Generate Custom Realistic Genotypes

Generate NEW synthetic genotypes with realistic LD structure.
Requires Singularity or Docker (downloads ~10GB reference data).

from synthlab import HAPNESTRunner, HAPNESTConfig

# Initialize runner (downloads container + reference data)
runner = HAPNESTRunner()
runner.initialize()

# Configure generation
config = HAPNESTConfig(
    n_samples=1000,        # Number of individuals
    chromosome=22,         # Single chromosome (1-22 or 'all')
    superpopulation='EUR', # Ancestry: AFR, AMR, CSA, EAS, EUR, MID
    seed=42,               # Reproducibility
)

# Generate!
result = runner.run(config)
# Output in: {result['output_dir']}

Code: https://github.com/intervene-EU-H2020/synthetic_data


In [ ]:
# Option 3: Quick random genotypes (NO LD - for testing only!)
# Use this ONLY for testing pipelines, NOT for real analysis

print("⚠️  WARNING: Random genotypes have NO LD structure!")
print("   Only use for: testing pipelines, checking code runs")
print("   Do NOT use for: GWAS, PRS, fine-mapping, or any real analysis")
print()

geno_dir = sl.generate_random_genotypes(
    n_samples=100,    # Number of individuals
    n_variants=1000,  # Number of SNPs
    seed=42,
)

print(f"\nFiles generated in {geno_dir}:")
for f in sorted(geno_dir.iterdir()):
    if f.name.startswith("demo_"):
        size = f.stat().st_size / 1024
        print(f"  • {f.name} ({size:.1f} KB)")

---
## 3. Medical Imaging

### Imaging Formats

| Format | Used For | Features |
|--------|----------|----------|
| **DICOM** | Radiology (CT, MRI, X-ray) | Rich metadata, 3D volumes |
| **NIfTI** | Neuroimaging research | Simpler format, good for analysis |
| **PNG/TIFF** | Pathology, dermatology | Standard image formats |

### Available Datasets

SynthLab catalogs publicly available imaging datasets. Access types:
- 🟢 **Open**: Download immediately
- 🟡 **Registration**: Free account required
- 🔴 **DUA**: Data use agreement required

In [21]:
# Browse the imaging catalog
sl.print_dataset_catalog()


Medical Imaging Dataset Catalog

CT
----------------------------------------
  🟢 LIDC-IDRI
     Lung Image Database Consortium - CT scans with lung nodule a...
     Size: 125.0 GB | Format: DICOM | Access: open
     Images: 1,018 | Patients: 1,010


Histopathology
----------------------------------------
  🟢 MHIST
     Minimalist Histopathology Image Analysis Dataset - H&E stain...
     Size: 0.3 GB | Format: PNG | Access: open
     Images: 3,152

  🟢 PanNuke
     Pan-cancer histology dataset for nuclei instance segmentatio...
     Size: 2.0 GB | Format: PNG/NPY | Access: open
     Images: 7,753

  🟡 CAMELYON16
     Detection of cancer metastases in lymph node WSI...
     Size: 700.0 GB | Format: TIFF (WSI) | Access: registration
     Images: 400

  🟡 CAMELYON17
     Automated detection and classification of breast cancer meta...
     Size: 2500.0 GB | Format: TIFF (WSI) | Access: registration
     Images: 1,000 | Patients: 200

  🟢 BCSS
     Breast Cancer Semantic Segmentation datase

In [22]:
# Find open-access datasets
open_datasets = sl.list_imaging_datasets(access_type="open")

print("Open-access datasets (no registration):")
print("=" * 50)

for name, ds in open_datasets.items():
    print(f"\n{ds.name}")
    print(f"  Modality: {ds.modality}")
    print(f"  Images: {ds.n_images:,}" if ds.n_images else "  Images: varies")
    print(f"  Size: {ds.size_gb:.1f} GB" if ds.size_gb else "  Size: varies")

Open-access datasets (no registration):

MHIST
  Modality: Histopathology
  Images: 3,152
  Size: 0.3 GB

PanNuke
  Modality: Histopathology
  Images: 7,753
  Size: 2.0 GB

BCSS
  Modality: Histopathology
  Images: 151
  Size: 1.5 GB

LIDC-IDRI
  Modality: CT
  Images: 1,018
  Size: 125.0 GB

ChestX-ray14
  Modality: X-ray
  Images: 112,120
  Size: 42.0 GB

SNOW
  Modality: Histopathology (Synthetic)
  Images: varies
  Size: varies


---
## Summary

```python
import synthlab as sl

# ═══════════════════════════════════════════════════════════
# EHR Generation
# ═══════════════════════════════════════════════════════════
sl.list_synthea_datasets()           # List pre-generated datasets
sl.download_dataset('synthea1k')     # Download pre-generated
runner = sl.SyntheaRunner()          # Generate new patients (requires Java)

# ═══════════════════════════════════════════════════════════
# Genomics Generation (HAPNEST)
# ═══════════════════════════════════════════════════════════
# Option 1: Download pre-generated (REALISTIC LD) ✅ Recommended
sl.download_hapnest()

# Option 2: Generate new (REALISTIC LD) - requires Singularity/Docker
from synthlab import HAPNESTRunner, HAPNESTConfig
runner = HAPNESTRunner()
config = HAPNESTConfig(n_samples=1000, chromosome=22, superpopulation='EUR')
runner.run(config)

# Option 3: Quick random genotypes (NO LD - testing only!)
sl.generate_random_genotypes(n_samples=100, n_variants=1000)

# ═══════════════════════════════════════════════════════════
# Imaging Catalog
# ═══════════════════════════════════════════════════════════
sl.print_dataset_catalog()           # Browse datasets
sl.list_imaging_datasets()           # Filter by modality/access
```

### Next Steps

- [Using Multimodal Data](./Coherent_MultimodalDataset.ipynb) - Work with linked EHR + Imaging + Genomics
- [AI Clinical Notes](./MedGemma_SOAP_Notes.ipynb) - Generate SOAP notes with MedGemma